In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# %%
# ================================
# 1. LOAD DATA (for both EDA & Preprocessing)
# ================================
df = pd.read_csv('heart_disease_uci.csv')
print("=" * 60)
print("HEART DISEASE UCI - FULL PIPELINE")
print("=" * 60)

# %%
# ================================
# 2. EXPLORATORY DATA ANALYSIS (EDA)
# ================================
print("\n" + "=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 60)

# Basic Info
print(f"\n[1] Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\n[2] Column Data Types:")
print(df.dtypes.to_string())

print("\n[3] First 5 Rows:")
print(df.head().to_string())

print("\n[4] Missing Values (count & %):")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].to_string())

print("\n[5] Descriptive Statistics (numeric):")
print(df.describe().to_string())

print("\n[6] Target Variable Distribution (num):")
print(df['num'].value_counts().to_string())

print("\n[7] Categorical Unique Values:")
cat_cols = ['sex', 'dataset', 'cp', 'restecg', 'slope', 'thal']
for col in cat_cols:
    print(f"  {col}: {df[col].dropna().unique().tolist()}")

print("\n[8] Duplicate Rows:", df.duplicated().sum())

# %%
# EDA Visualisations
sns.set_theme(style='whitegrid', palette='muted')

# Fig 1 – Missing value heatmap
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(df.isnull().T, cbar=False, yticklabels=df.columns, cmap='Reds', ax=ax)
ax.set_title('Missing Value Heatmap', fontsize=13, fontweight='bold')
ax.set_xlabel('Sample Index')
plt.tight_layout()
plt.savefig('eda_missing_heatmap.png', dpi=150)
plt.close()

# Fig 2 – Target distribution
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
val_counts = df['num'].value_counts().sort_index()
labels_full = ['0 – No Disease', '1 – Mild', '2 – Moderate', '3 – Severe', '4 – Very Severe']
colors = sns.color_palette('Set2', 5)
axes[0].bar(val_counts.index, val_counts.values, color=colors, edgecolor='black', linewidth=0.6)
axes[0].set_title('Target (num) Distribution', fontweight='bold')
axes[0].set_xlabel('Heart Disease Severity (0–4)')
axes[0].set_ylabel('Count')
for i, v in zip(val_counts.index, val_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=9)

binary = df['num'].apply(lambda x: 0 if x == 0 else 1)
bc = binary.value_counts()
axes[1].pie(bc, labels=['No Disease (0)', 'Disease (1)'], autopct='%1.1f%%',
            colors=['#66b3ff', '#ff9999'], startangle=140)
axes[1].set_title('Binary Target Split', fontweight='bold')
plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150)
plt.close()

# Fig 3 – Numeric feature distributions
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak']
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_xlabel('')
axes[-1].set_visible(False)
fig.suptitle('Numeric Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_numeric_distributions.png', dpi=150)
plt.close()

# Fig 4 – Correlation heatmap
num_df = df[['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca', 'num']].dropna()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(num_df.corr(), dtype=bool))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix (Numeric Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png', dpi=150)
plt.close()

# Fig 5 – Categorical vs target
cat_cols_plot = ['sex', 'cp', 'dataset']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(cat_cols_plot):
    ct = pd.crosstab(df[col], df['num'].apply(lambda x: 'Disease' if x > 0 else 'No Disease'))
    ct.plot(kind='bar', ax=axes[i], colormap='Set2', edgecolor='black', linewidth=0.5, rot=30)
    axes[i].set_title(f'{col} vs Target', fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=8)
fig.suptitle('Categorical Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_categorical_vs_target.png', dpi=150)
plt.close()

print("\n[✓] EDA complete. Plots saved:")
for f in ['eda_missing_heatmap.png', 'eda_target_distribution.png',
          'eda_numeric_distributions.png', 'eda_correlation_heatmap.png',
          'eda_categorical_vs_target.png']:
    print(f"    {f}")


HEART DISEASE UCI - FULL PIPELINE

EXPLORATORY DATA ANALYSIS

[1] Dataset Shape: 920 rows × 16 columns

[2] Column Data Types:
id            int64
age           int64
sex          object
dataset      object
cp           object
trestbps    float64
chol        float64
fbs          object
restecg      object
thalch      float64
exang        object
oldpeak     float64
slope        object
ca          float64
thal         object
num           int64

[3] First 5 Rows:
   id  age     sex    dataset               cp  trestbps   chol    fbs         restecg  thalch  exang  oldpeak        slope   ca               thal  num
0   1   63    Male  Cleveland   typical angina     145.0  233.0   True  lv hypertrophy   150.0  False      2.3  downsloping  0.0       fixed defect    0
1   2   67    Male  Cleveland     asymptomatic     160.0  286.0  False  lv hypertrophy   108.0   True      1.5         flat  3.0             normal    2
2   3   67    Male  Cleveland     asymptomatic     120.0  229.0  False  lv 